In [1]:
from __future__ import annotations

import copy
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find repo root containing /src")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import evaluate_count_predictions
from src.models.feature_sets import BASE_TABULAR_FEATURES, GCMT_FEATURES, QUALITY_FEATURES
from src.models.input_layer import InputConfig, load_modeling_splits, prepare_tabular_inputs
from src.utils.paths import METRICS_DIR

DATASET_NAME = "earthquake_aftershock_v2_gcmt"
MODEL_NAME   = "zi_lognormal_dl"
TARGETS      = ["n_aftershocks_24h", "n_aftershocks_72h"]
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(42)
print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {DEVICE}")

Project root : /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10
Device       : cpu


In [2]:
splits = load_modeling_splits(dataset_name=DATASET_NAME)

for split_name, df in splits.items():
    for t in TARGETS:
        s = df[t]
        print(f"{split_name:5s} {t}: "
              f"mean={s.mean():.1f}  median={s.median():.0f}  "
              f"zeros={( s==0).mean():.1%}  max={s.max():.0f}")

train n_aftershocks_24h: mean=6.7  median=0  zeros=55.7%  max=1253
train n_aftershocks_72h: mean=10.9  median=1  zeros=49.6%  max=1643
val   n_aftershocks_24h: mean=13.5  median=0  zeros=52.2%  max=282
val   n_aftershocks_72h: mean=20.5  median=1  zeros=46.5%  max=392
test  n_aftershocks_24h: mean=9.6  median=0  zeros=53.1%  max=389
test  n_aftershocks_72h: mean=14.4  median=1  zeros=47.0%  max=605


In [3]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    has_gcmt = (
        df["has_gcmt"].fillna(False)
        if "has_gcmt" in df.columns
        else pd.Series(False, index=df.index)
    )

    df["depth_shallow"]      = (df["trigger_depth_km"] < 70).astype(float)
    df["depth_intermediate"] = df["trigger_depth_km"].between(70, 300).astype(float)
    df["depth_deep"]         = (df["trigger_depth_km"] > 300).astype(float)
    df["log_magnitude"]      = np.log10(df["trigger_magnitude"].clip(lower=1e-6))
    df["mag_depth_ratio"]    = df["trigger_magnitude"] / (df["trigger_depth_km"] + 1)
    df["log_scalar_moment"]  = np.where(
        has_gcmt & df["gcmt_scalar_moment"].notna(),
        np.log10(df["gcmt_scalar_moment"].clip(lower=1e-10)), np.nan,
    )
    df["sin_month"]     = np.sin(2 * np.pi * df["trigger_month"]     / 12)
    df["cos_month"]     = np.cos(2 * np.pi * df["trigger_month"]     / 12)
    df["sin_hour"]      = np.sin(2 * np.pi * df["trigger_hour"]      / 24)
    df["cos_hour"]      = np.cos(2 * np.pi * df["trigger_hour"]      / 24)
    df["sin_dayofyear"] = np.sin(2 * np.pi * df["trigger_dayofyear"] / 365)
    df["cos_dayofyear"] = np.cos(2 * np.pi * df["trigger_dayofyear"] / 365)
    df["ring_of_fire"]  = (
        (np.abs(df["trigger_longitude"]) > 130)
        & df["trigger_latitude"].between(-60, 60)
    ).astype(float)

    def _rake_regimes(rake):
        if pd.isna(rake): return np.nan, np.nan, np.nan
        r = rake % 360
        ss = min(abs(r), abs(r - 180), abs(r - 360))
        return float(ss < 45), float(45 <= r <= 135), float(225 <= r <= 315)

    regimes = df["rake"].apply(
        lambda r: _rake_regimes(r) if pd.notna(r) else (np.nan, np.nan, np.nan)
    )
    df["is_strike_slip"] = regimes.apply(lambda x: x[0]).where(has_gcmt)
    df["is_reverse"]     = regimes.apply(lambda x: x[1]).where(has_gcmt)
    df["is_normal"]      = regimes.apply(lambda x: x[2]).where(has_gcmt)
    df["sin_dip"]        = np.sin(np.radians(df["dip"].where(has_gcmt)))

    e1 = df["gcmt_eig1"].where(has_gcmt)
    e2 = df["gcmt_eig2"].where(has_gcmt)
    e3 = df["gcmt_eig3"].where(has_gcmt)
    denom = (e1.abs() + e3.abs()).replace(0, np.nan)
    df["clvd_fraction"]       = (2 * e2.abs() / denom).where(has_gcmt)
    df["centroid_depth_diff"] = (df["gcmt_depth_km"] - df["trigger_depth_km"]).where(has_gcmt)
    df["log_half_duration"]   = np.log1p(df["gcmt_half_duration_sec"].where(has_gcmt))
    df["mag_diff_abs"]        = df["gcmt_mag_diff"].abs().where(has_gcmt)
    df["moment_exponent_centered"] = (df["gcmt_moment_exponent"].where(has_gcmt) - 24.0)

    prior_24h = df["prior_global_event_count_24h"]
    prior_7d  = df["prior_global_event_count_7d"]
    df["log_prior_24h"]           = np.log1p(prior_24h)
    df["log_prior_7d"]            = np.log1p(prior_7d)
    df["seismicity_acceleration"] = prior_24h / (prior_7d / 7.0).replace(0, np.nan)
    df["mag_x_shallow"]           = df["trigger_magnitude"] * df["depth_shallow"]
    df["mag_x_log_prior"]         = df["trigger_magnitude"] * np.log1p(prior_24h)
    return df


ENGINEERED_FEATURES = [
    "depth_shallow","depth_intermediate","depth_deep",
    "log_magnitude","mag_depth_ratio","log_scalar_moment",
    "sin_month","cos_month","sin_hour","cos_hour","sin_dayofyear","cos_dayofyear",
    "ring_of_fire","is_strike_slip","is_reverse","is_normal",
    "sin_dip","clvd_fraction","centroid_depth_diff","log_half_duration","mag_diff_abs",
    "moment_exponent_centered",
    "log_prior_24h","log_prior_7d","seismicity_acceleration",
    "mag_x_shallow","mag_x_log_prior",
]

_seen: set[str] = set()
FULL_FEATURE_SET: list[str] = []
for f in BASE_TABULAR_FEATURES + QUALITY_FEATURES + GCMT_FEATURES + ENGINEERED_FEATURES:
    if f not in _seen:
        FULL_FEATURE_SET.append(f)
        _seen.add(f)

splits_eng = {name: engineer_features(df) for name, df in splits.items()}
print(f"Features requested: {len(FULL_FEATURE_SET)}")

Features requested: 83


In [4]:
prepared: dict[str, object] = {}

for target in TARGETS:
    config = InputConfig(
        feature_cols=FULL_FEATURE_SET,
        target_col=target,
        missing_strategy="median",
        scale=True,
        allow_missing_optional=True,
        drop_rows_with_missing_target=True,
    )
    inputs = prepare_tabular_inputs(config=config, splits=splits_eng)
    prepared[target] = inputs
    print(f"{target} — X_train: {inputs.X_train.shape}  "
          f"y_train mean: {inputs.y_train.mean():.2f}")

n_aftershocks_24h — X_train: (23031, 83)  y_train mean: 6.67
n_aftershocks_72h — X_train: (23031, 83)  y_train mean: 10.86


In [5]:
class ZeroInflatedResBlock(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.2) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.LayerNorm(dim),
        )
        self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(x + self.net(x))


class ZeroInflatedLogNormalNet(nn.Module):
    """
    Two-head network for zero-inflated count prediction.
    Shared residual trunk → separate zero-probability and log-count heads.
    """
    def __init__(
        self,
        n_features: int,
        hidden_dim: int   = 256,
        n_blocks:   int   = 4,
        dropout:    float = 0.2,
    ) -> None:
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.trunk = nn.Sequential(
            *[ZeroInflatedResBlock(hidden_dim, dropout) for _ in range(n_blocks)]
        )
        # Zero-inflation head: P(count = 0)
        self.zero_head  = nn.Linear(hidden_dim, 1)
        # Count head: predicts log1p(count) — only meaningful for non-zero events
        self.count_head = nn.Linear(hidden_dim, 1)

    def forward(
        self, x: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.trunk(self.input_proj(x))
        p_zero    = torch.sigmoid(self.zero_head(h)).squeeze(1)   # (B,) in [0,1]
        log_count = self.count_head(h).squeeze(1)                 # (B,) unbounded
        return p_zero, log_count


def zi_lognormal_loss(
    p_zero:    torch.Tensor,
    log_count: torch.Tensor,
    y_true:    torch.Tensor,
    alpha:     float = 0.5,
) -> torch.Tensor:
    """
    Combined zero-inflation + log-count loss.

    Args:
        p_zero:    P(count=0) predicted by zero head
        log_count: predicted log1p(count)
        y_true:    raw count targets
        alpha:     weight balancing zero-head vs count-head losses
    """
    is_zero = (y_true == 0).float()

    # Zero head: BCE against the zero/non-zero indicator
    zero_loss = nn.BCELoss()(p_zero, is_zero)

    # Count head: MSE on log1p scale, only for non-zero events
    log_true     = torch.log1p(y_true)
    nonzero_mask = (y_true > 0)
    if nonzero_mask.sum() > 0:
        count_loss = nn.MSELoss()(log_count[nonzero_mask], log_true[nonzero_mask])
    else:
        count_loss = torch.tensor(0.0)

    return alpha * zero_loss + (1.0 - alpha) * count_loss


def predict_counts(
    model: ZeroInflatedLogNormalNet,
    X: pd.DataFrame,
    device: str,
    batch_size: int = 1024,
) -> np.ndarray:
    """
    Final count prediction: (1 - P_zero) * expm1(log_count).
    The zero probability gates how much of the count prediction survives.
    """
    model.eval()
    all_preds = []
    X_t = torch.tensor(X.values.astype(np.float32))
    loader = DataLoader(TensorDataset(X_t), batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for (Xb,) in loader:
            p_zero, log_count = model(Xb.to(device))
            # Predicted count = expected value given the zero-inflation mixture
            pred = (1.0 - p_zero) * torch.expm1(log_count.clamp(min=0.0))
            all_preds.append(pred.cpu().numpy())
    return np.concatenate(all_preds).clip(min=0.0)


# Sanity check
with torch.no_grad():
    _n = len(prepared["n_aftershocks_24h"].feature_cols)
    _m = ZeroInflatedLogNormalNet(_n)
    _x = torch.randn(4, _n); _y = torch.tensor([0.0, 3.0, 0.0, 12.0])
    _pz, _lc = _m(_x)
    _loss = zi_lognormal_loss(_pz, _lc, _y)
    n_params = sum(p.numel() for p in _m.parameters())
    print(f"Architecture OK — {_n} features, {n_params:,} parameters, loss={_loss.item():.4f}")
    del _m, _x, _y, _pz, _lc

Architecture OK — 83 features, 552,962 parameters, loss=3.5817


In [6]:
NET_PARAMS = dict(
    hidden_dim = 256,
    n_blocks   = 4,
    dropout    = 0.2,
)

TRAIN_PARAMS = dict(
    lr           = 3e-4,
    weight_decay = 1e-4,
    batch_size   = 512,
    max_epochs   = 200,
    patience     = 25,
    alpha        = 0.5,   # balance between zero-head and count-head loss
)

trained_models: dict[str, ZeroInflatedLogNormalNet] = {}

for target in TARGETS:
    print(f"\n===== Training {target} =====")
    inp = prepared[target]
    n_feats = inp.X_train.shape[1]

    Xtr = torch.tensor(inp.X_train.values.astype(np.float32))
    ytr = torch.tensor(inp.y_train.values.astype(np.float32))
    Xva = torch.tensor(inp.X_val.values.astype(np.float32))
    yva = torch.tensor(inp.y_val.values.astype(np.float32))

    train_loader = DataLoader(
        TensorDataset(Xtr, ytr),
        batch_size=TRAIN_PARAMS["batch_size"], shuffle=True, drop_last=True,
    )

    model = ZeroInflatedLogNormalNet(n_feats, **NET_PARAMS).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_PARAMS["lr"],
        weight_decay=TRAIN_PARAMS["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TRAIN_PARAMS["max_epochs"], eta_min=1e-6,
    )

    best_val_mae = float("inf")
    best_state   = None
    no_improve   = 0

    for epoch in range(1, TRAIN_PARAMS["max_epochs"] + 1):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            p_zero, log_count = model(Xb)
            loss = zi_lognormal_loss(p_zero, log_count, yb, TRAIN_PARAMS["alpha"])
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        scheduler.step()

        # Validate on log1p scale (more informative than raw MAE for tuning)
        val_preds = predict_counts(model, inp.X_val, DEVICE)
        val_mae   = mean_absolute_error(inp.y_val.values, val_preds)

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state   = copy.deepcopy(model.state_dict())
            no_improve   = 0
        else:
            no_improve += 1

        if epoch % 25 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d} | val_MAE={val_mae:.3f} | "
                  f"best={best_val_mae:.3f} | lr={scheduler.get_last_lr()[0]:.2e}")

        if no_improve >= TRAIN_PARAMS["patience"]:
            print(f"  Early stopping at epoch {epoch}. Best val MAE: {best_val_mae:.3f}")
            break

    model.load_state_dict(best_state)
    trained_models[target] = model
    print(f"  Final test MAE: {mean_absolute_error(inp.y_test.values, predict_counts(model, inp.X_test, DEVICE)):.3f}")


===== Training n_aftershocks_24h =====
  Epoch   1 | val_MAE=11.929 | best=11.929 | lr=3.00e-04
  Epoch  25 | val_MAE=12.419 | best=11.265 | lr=2.89e-04
  Early stopping at epoch 28. Best val MAE: 11.265
  Final test MAE: 8.436

===== Training n_aftershocks_72h =====
  Epoch   1 | val_MAE=18.591 | best=18.591 | lr=3.00e-04
  Epoch  25 | val_MAE=19.006 | best=17.732 | lr=2.89e-04
  Early stopping at epoch 29. Best val MAE: 17.732
  Final test MAE: 11.988


In [9]:
summary_rows = []
import re

for target in TARGETS:
    inp     = prepared[target]
    model   = trained_models[target]
    horizon = int(re.search(r"(\d+)h$", target).group(1))

    for split_name, X_df, y_s in [
        ("train", inp.X_train, inp.y_train),
        ("val",   inp.X_val,   inp.y_val),
        ("test",  inp.X_test,  inp.y_test),
    ]:
        y_pred = predict_counts(model, X_df, DEVICE)
        m = evaluate_count_predictions(y_true=y_s, y_pred=y_pred)
        summary_rows.append({
            "model_name": MODEL_NAME,
            "split":      split_name,
            "horizon":    horizon,
            **m,
        })

_split_order = {"train": 0, "val": 1, "test": 2}
metrics_df = (
    pd.DataFrame(summary_rows)
    .assign(_o=lambda d: d["split"].map(_split_order))
    .sort_values(["model_name", "horizon", "_o"])
    .drop(columns="_o")
    .reset_index(drop=True)
)

print(metrics_df[["model_name","horizon","split","mae","rmse",
                  "mean_true","mean_pred"]].to_string(index=False))

     model_name  horizon split       mae      rmse  mean_true  mean_pred
zi_lognormal_dl       24 train  5.903234 22.324995   6.671052   6.032230
zi_lognormal_dl       24   val 11.265141 35.597576  13.459289   6.206036
zi_lognormal_dl       24  test  8.436308 24.365572   9.626815   6.448791
zi_lognormal_dl       72 train  8.487332 35.214431  10.858929   8.272923
zi_lognormal_dl       72   val 17.731509 55.469642  20.534977   7.311108
zi_lognormal_dl       72  test 11.987830 36.498051  14.423570   8.315346


In [11]:
import math

print("=== Log1p-scale MAE (what the model optimises) ===")
for target in TARGETS:
    inp     = prepared[target]
    model   = trained_models[target]
    horizon = int(re.search(r"(\d+)h$", target).group(1))
    for split_name, X_df, y_s in [
        ("val",  inp.X_val,  inp.y_val),
        ("test", inp.X_test, inp.y_test),
    ]:
        y_pred     = predict_counts(model, X_df, DEVICE)
        log_mae    = mean_absolute_error(np.log1p(y_s.values), np.log1p(y_pred.clip(0)))
        naive_mean = mean_absolute_error(np.log1p(y_s.values),
                                         np.full_like(y_s.values, np.log1p(y_s.mean())))
        print(f"  {split_name:5s} horizon={horizon}h | "
              f"log_MAE={log_mae:.4f} | "
              f"naive_mean_log_MAE={naive_mean:.4f} | "
              f"gain={((naive_mean - log_mae) / naive_mean * 100):+.1f}%")

print("\n=== Raw-scale (dominated by extreme events — expected to be high) ===")
print(metrics_df[["horizon","split","mae","rmse","mean_true","mean_pred"]].to_string(index=False))
print("\nNote: raw RMSE is heavily influenced by the top 1% of extreme sequences.")
print("Log-scale MAE is the meaningful metric for count models on this distribution.")

=== Log1p-scale MAE (what the model optimises) ===
  val   horizon=24h | log_MAE=0.7944 | naive_mean_log_MAE=1.6084 | gain=+50.6%
  test  horizon=24h | log_MAE=0.8041 | naive_mean_log_MAE=1.6003 | gain=+49.8%
  val   horizon=72h | log_MAE=0.9160 | naive_mean_log_MAE=2.2108 | gain=+58.6%
  test  horizon=72h | log_MAE=0.8811 | naive_mean_log_MAE=1.5923 | gain=+44.7%

=== Raw-scale (dominated by extreme events — expected to be high) ===
 horizon split       mae      rmse  mean_true  mean_pred
      24 train  5.903234 22.324995   6.671052   6.032230
      24   val 11.265141 35.597576  13.459289   6.206036
      24  test  8.436308 24.365572   9.626815   6.448791
      72 train  8.487332 35.214431  10.858929   8.272923
      72   val 17.731509 55.469642  20.534977   7.311108
      72  test 11.987830 36.498051  14.423570   8.315346

Note: raw RMSE is heavily influenced by the top 1% of extreme sequences.
Log-scale MAE is the meaningful metric for count models on this distribution.


In [13]:
prediction_rows = []

for target in TARGETS:
    inp     = prepared[target]
    model   = trained_models[target]
    horizon = int(re.search(r"(\d+)h$", target).group(1))

    for split_name, X_df, y_s, raw_df in [
        ("train", inp.X_train, inp.y_train, splits_eng["train"]),
        ("val",   inp.X_val,   inp.y_val,   splits_eng["val"]),
        ("test",  inp.X_test,  inp.y_test,  splits_eng["test"]),
    ]:
        y_pred = predict_counts(model, X_df, DEVICE)
        prediction_rows.append(pd.DataFrame({
            "trigger_event_id": raw_df.loc[X_df.index, "trigger_event_id"].values,
            "split":            split_name,
            "horizon":          horizon,
            "model_name":       MODEL_NAME,
            "y_true":           y_s.values,
            "y_pred":           y_pred,       # y_pred (not y_prob) for count tasks
        }))

predictions_df = pd.concat(prediction_rows, ignore_index=True)

METRICS_DIR.mkdir(parents=True, exist_ok=True)
predictions_df.to_csv(METRICS_DIR / f"{MODEL_NAME}_predictions.csv", index=False)
metrics_df.to_csv(    METRICS_DIR / f"{MODEL_NAME}_metrics.csv",     index=False)

print(f"Predictions → {METRICS_DIR / f'{MODEL_NAME}_predictions.csv'}")
print(f"Metrics     → {METRICS_DIR / f'{MODEL_NAME}_metrics.csv'}")
print(f"Total rows  : {len(predictions_df)}")

Predictions → /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/zi_lognormal_dl_predictions.csv
Metrics     → /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/zi_lognormal_dl_metrics.csv
Total rows  : 56576
